# Hanafuda Card Detection — Quickstart Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tarumino3/Hanafuda-YOLO/blob/main/notebooks/01_quickstart_demo.ipynb)

This notebook lets you try Hanafuda card detection.
Upload any photo of Hanafuda cards and the model will detect and label each one.

| Model | Classes | mAP50 | mAP50-95 |
|-------|---------|-------|----------|
| YOLO11n | 36 | **99.5%** | **93.5%** |

## Step 1 — Install & Clone

In [ ]:
# Clone the repository and install dependencies
!git clone https://github.com/tarumino3/Hanafuda-YOLO.git
%cd Hanafuda-YOLO
!pip install -r requirements.txt --quiet

## Step 2 — Upload Your Image

Upload a photo of Hanafuda cards from your device.
The cards can be overlapping or at any angle — just like a real game.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
# Resolve to absolute path — avoids ambiguity when ultralytics resolves the source
image_path = Path(next(iter(uploaded))).resolve()
print(f"Uploaded: {image_path}  ({image_path.stat().st_size // 1024} KB)")

## Step 3 — Detect Cards

In [ ]:
import sys
sys.path.insert(0, '.')

from IPython.display import display
from PIL import Image
from ultralytics import YOLO

# Load model
model = YOLO('models/best.pt')

# predict() handles letterbox inverse transforms internally
results = model(str(image_path), conf=0.25, device='cpu', verbose=False)

# result.plot() draws boxes on the original image — no custom coordinate math needed
annotated = Image.fromarray(results[0].plot()[..., ::-1])  # BGR → RGB
display(annotated)

## Step 4 — Inspect Results

In [ ]:
r = results[0]
boxes = r.boxes

print(f"Detected {len(boxes)} card(s)\n")
print(f"{'Card':<25} {'Confidence':>10}")
print("-" * 37)

detections = [
    (r.names[int(cls)], float(conf))
    for cls, conf in zip(boxes.cls, boxes.conf)
]
for name, score in sorted(detections, key=lambda x: x[1], reverse=True):
    bar = "█" * int(score * 20)
    print(f"{name:<25} {score:>6.1%}  {bar}")

## (Optional) Save Annotated Image

In [ ]:
output_path = str(image_path.parent / (image_path.stem + '_detected.jpg'))
annotated.save(output_path)

files.download(output_path)
print(f"Saved: {output_path}")